## Load Libraries & Data

In [ ]:
!pip install gensim

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine
from gensim.models import Word2Vec

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

In [ ]:
all_recipes = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2025/2025-09-16/all_recipes.csv')
cuisines = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2025/2025-09-16/cuisines.csv')

print(f"Initial shapes: all_recipes {all_recipes.shape}, cuisines {cuisines.shape}")

## Data Overview

In [ ]:
print(all_recipes.columns.tolist())
print(all_recipes.head(1))

In [ ]:
all_recipes['url'].describe()    # Every recipe is unique

In [ ]:
# Missing values check
miss = all_recipes.isnull().sum()
miss = miss[miss > 0].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(miss.index, miss.values, color='#5B8DB8', edgecolor='white')
ax.set_xlabel('Missing count')
ax.set_title('Missing Values per Column')
for i, v in enumerate(miss.values):
    ax.text(v + 5, i, str(v), va='center', fontsize=9)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Ratings distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(all_recipes['avg_rating'].dropna(), bins=25, kde=True,
             color='#5B8DB8', ax=axes[0])
axes[0].set_title('Average Ratings')
axes[0].set_xlabel('Average rating')
axes[0].set_ylabel('')

sns.histplot(all_recipes['total_ratings'].dropna().clip(upper=500), bins=30, kde=True,
             color='#E07B54', ax=axes[1])
axes[1].set_title('Total Ratings (clipped at 500)')
axes[1].set_xlabel('Total rating')
axes[1].set_ylabel('')

sns.despine()
plt.tight_layout()
plt.show()

ratings_before = all_recipes['avg_rating'].dropna()

In [ ]:
# Publication years
all_recipes['year'] = pd.to_datetime(all_recipes['date_published']).dt.year
fig, ax = plt.subplots(figsize=(8,4))
all_recipes['year'].value_counts().sort_index().plot(marker='o')
plt.title('Recipes Published per Year')
ax.set_xlabel('Year')
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Nutrition distribution overview
nutrients = ['calories', 'fat', 'carbs', 'protein']

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
colors = ['#5B8DB8', '#E07B54', '#6DBF9E', '#A78BCA']

for ax, col, color in zip(axes, nutrients, colors):
    data = all_recipes[col].dropna().clip(upper=all_recipes[col].quantile(0.97))
    sns.histplot(data, bins=30, kde=True, color=color, ax=ax)
    ax.set_title(col.capitalize())
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.axvline(data.median(), color='black', linestyle='--', linewidth=1, label=f'median {data.median():.0f}')
    ax.legend(fontsize=8)

sns.despine()
plt.suptitle('Nutritional Distributions (clipped at 97th percentile)', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Prep, cook and total time
time_cols = ['prep_time', 'cook_time', 'total_time']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, time_cols):
    data = all_recipes[col].dropna()

    # Print outlier info
    zeros = (data == 0).sum()
    print(f"\n{col}:")
    print(f"  Zero values: {zeros} ({(zeros/len(data))*100:.2f}%)")
    print(f"  Mean: {data.mean():.0f} min")
    print(f"  Median: {data.median():.0f} min")
    print(f"  Max: {data.max():.0f} min")
    print(f"  99th percentile: {data.quantile(0.99):.0f} min")

    # Plot histogram with values capped at 99th percentile for readability
    cap = data.quantile(0.99)
    clipped = data.clip(upper=cap)

    sns.histplot(clipped, bins=30, kde=True, color='#5B8DB8', ax=ax)
    ax.axvline(data.median(), color='red', linestyle='--', label=f'Median = {data.median():.0f}')
    ax.set_title(f'{col.replace("_", " ").capitalize()} (capped at 99th pct)')
    ax.set_xlabel('Minutes')
    ax.set_ylabel('')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Correlation
num_cols = ['avg_rating', 'total_time', 'servings', 'calories', 'fat', 'carbs', 'protein']

# Compute correlation matrix
corr = all_recipes[num_cols].corr()

# Plot heatmap
plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))  # optional: mask upper triangle
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, mask=mask)
plt.title('Correlation Matrix of Numeric Features')
plt.tight_layout()
plt.show()

## Data Cleaning

In [ ]:
print(f"Original shape of the database: {all_recipes.shape}")

In [ ]:
# Remove pet recipes
all_recipes = all_recipes[~all_recipes["name"].str.contains("dog|cat|puppy|puppies|pet", case=False, na=False)]
print(f"After removing pet recipes: {len(all_recipes)}")

In [ ]:
# Drop rows with missing ingredients
all_recipes = all_recipes.dropna(subset=['ingredients'])
print(f"After dropping missing ingredients: {len(all_recipes)}")

In [ ]:
# Fill missing ratings with median
median_rating = all_recipes['avg_rating'].median()
all_recipes['avg_rating'] = all_recipes['avg_rating'].fillna(median_rating)
print(f"Missing ratings after cleaning: {all_recipes['avg_rating'].isna().sum()}")

# Drop low rated recipes (<3.5)
all_recipes = all_recipes[all_recipes['avg_rating'] >= 3.5]
print(f"After imputing ratings and dropping <3.5: {len(all_recipes)}")
ratings_after = all_recipes['avg_rating']

In [ ]:
# Remove extreme nutritional outliers (above 99th percentile in any nutrient)
nutrients = ['calories', 'fat', 'carbs', 'protein']
mask = pd.Series(True, index=all_recipes.index)
for col in nutrients:
    cap = all_recipes[col].quantile(0.99)
    mask &= (all_recipes[col].fillna(0) <= cap)
all_recipes = all_recipes[mask]
print(f"After removing nutritional outliers: {len(all_recipes)}")

In [ ]:
# Cap all time columns at their 99th percentile
for col in ['prep_time', 'cook_time', 'total_time']:
    cap = all_recipes[col].quantile(0.99)
    all_recipes[col] = all_recipes[col].clip(upper=cap)
    print(f"Capped {col} at {cap:.0f}")

In [ ]:
# Drop columns not needed for retrieval/ranking
all_recipes = all_recipes.drop(columns=[
    "author",
    "date_published",
    "calories",
    "fat",
    "carbs",
    "protein",
    "reviews",
    "year"
])

print(f"Remaining columns: {all_recipes.columns.tolist()}")
print(f"Final shape of the database: {all_recipes.shape}")

## Preprocessing

### Step 1 - Parse ingredients into clean lists


In [ ]:
def parse_ingredients(ing_str):
    """Convert raw ingredient string into a list of ingredient strings."""
    if pd.isna(ing_str):
        return []
    # Split ingredient lists by comma
    parts = ing_str.split(', ')
    # Pattern for a new ingredient: starts with digit, fraction, or common unit
    new_ing_pattern = re.compile(r'^[\d¼½¾⅓⅔⅕⅖⅗⅘⅙⅚⅛⅜⅝⅞]|^cup|^tablespoon|^teaspoon|^tbsp|^tsp|^ounce|^oz|^pound|^lb', re.IGNORECASE)
    merged = []
    for part in parts:
        if not merged:                              # If merged is empty, add the ingredient
            merged.append(part)
        elif new_ing_pattern.match(part.strip()):   # If new ingredient matches with the pattern, add it to the list
            merged.append(part)
        else:
            merged[-1] += ', ' + part               # If it doesn't, add it to the ingredient which was last added, with a comma
    return merged

In [ ]:
# Example on first recipe
raw_ingredients = all_recipes.iloc[0]['ingredients']
print("Raw ingredients string:\n", raw_ingredients, "\n")

parsed_list = parse_ingredients(raw_ingredients)
print("Parsed ingredient list (user‑friendly):")
for i, ing in enumerate(parsed_list, 1):
    print(f"  {i}. {ing}")

### Step 2 - Clean and normalise ingredient names

Used for retrieval and ranking

In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

non_ingredient_words = {
    # Measurement units
    'cup', 'cups', 'teaspoon', 'teaspoons', 'tablespoon', 'tablespoons', 'tbsp', 'tsp',
    'ounce', 'ounces', 'oz', 'pound', 'pounds', 'lb', 'lbs', 'gram', 'grams', 'g',
    'kilogram', 'kg', 'milliliter', 'ml', 'liter', 'l', 'pinch', 'dash', 'handful',
    'package', 'pack', 'can', 'bottle', 'jar', 'container', 'box', 'bag', 'inch',
    'liquid',

    # Preparation / adjectives
    'chopped', 'ground', 'fresh', 'white', 'taste', 'black', 'large', 'sliced',
    'minced', 'grated', 'shredded', 'diced', 'crushed', 'mashed', 'beaten',
    'cooked', 'raw', 'roasted', 'toasted', 'browned', 'melted', 'softened',
    'divided', 'reserved', 'warm', 'cold', 'room', 'temperature', 'fine', 'coarse',
    'whole', 'pure', 'extra', 'virgin', 'light', 'dark', 'low', 'fat', 'reduced',
    'sodium', 'unsalted', 'salted', 'sweet', 'sour', 'bitter', 'spicy', 'mild',
    'freshly', 'organic', 'ripe', 'frozen', 'canned', 'dried', 'fresh', 'minced',
    'grated', 'diced', 'sliced', 'crushed', 'mashed', 'pureed', 'blended', 'whipped',
    'needed', 'peeled', 'cubed', 'drained',

    # Other
    'plus', 'for', 'with', 'into', 'from', 'by', 'at', 'of', 'and', 'or', 'the', 'a', 'an',
    'each', 'every', 'all', 'both', 'neither', 'either', 'not', 'no', 'very', 'just', 'but',
    'so', 'then', 'now', 'also', 'too', 'like', 'use', 'add', 'mix', 'stir', 'pour', 'bake',
    'until', 'when', 'then', 'after', 'before', 'while', 'during', 'without', 'optional',
    'salt','pepper','salt pepper', 'water'
}

def clean_ingredient(ingredient_text):
    """Clean a single ingredient string."""
    # 1. If not a proper string, return empty
    if not isinstance(ingredient_text, str):
        return ""

    # 2. Lowercase and remove everything except a-z and spaces
    text = re.sub(r"[^a-z\s]", "", ingredient_text.lower())

    # 3. Split into words
    words = nltk.word_tokenize(text)

    # 4. Filter and lemmatize words
    cleaned = []
    for w in words:
        # Skip stopwords, non-ingredient words, and very short words
        if w in stop_words or w in non_ingredient_words or len(w) <= 2:
            continue
        # Lemmatize (default to noun)
        lemma = lemmatizer.lemmatize(w)
        cleaned.append(lemma)

    # 5. Join back into a string
    return " ".join(cleaned)

# Create a new column 'ingredients_list' as a list of cleaned ingredient strings
ingredients_lists = []
for raw_ing in all_recipes['ingredients']:
    # Split on commas
    parts = raw_ing.split(',')
    cleaned_parts = []
    for part in parts:
        cleaned_part = clean_ingredient(part.strip())
        if cleaned_part:          # only keep non-empty
            cleaned_parts.append(cleaned_part)
    ingredients_lists.append(cleaned_parts)

all_recipes['ingredients_list'] = ingredients_lists

# Remove recipes with empty list
all_recipes = all_recipes[all_recipes['ingredients_list'].apply(len) > 0].copy()
print(f"After cleaning ingredients: {len(all_recipes)} recipes remain")

# Create one big string of all cleaned ingredients per recipe (for the embedding model)
ingredient_texts = []
for lst in all_recipes['ingredients_list']:
    ingredient_texts.append(" ".join(lst))
all_recipes['ingredient_text'] = ingredient_texts

In [ ]:
# Example on first recipe
print(all_recipes['ingredients_list'].iloc[0][:5])

In [ ]:
# Normalising avg_rating for future ranking
max_rating = all_recipes['avg_rating'].max()
min_rating = all_recipes['avg_rating'].min()
all_recipes['rating_norm'] = (all_recipes['avg_rating'] - min_rating) / (max_rating - min_rating)

print(f"New column: rating_norm -> values between 0-1")

## Word embeddings

### 1. TF-IDF - Baseline

In [ ]:
docs = all_recipes["ingredients_list"].tolist()

# "pass-through" tokenizer and preprocessor
vectorizer = TfidfVectorizer(
    tokenizer=lambda x: x,       # x is already a list of tokens
    preprocessor=lambda x: x,    # do not change x, because the list is already cleaned
    token_pattern=None,          # disable default regex tokenizer
)

tfidf_matrix = vectorizer.fit_transform(docs)
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
feature_names = vectorizer.get_feature_names_out()
print(f"\nTF-IDF first 20 feature names:\n{feature_names[:20]}")

### 2. pre-trained SBERT - deep neural network based on the transformer architecture
- Transformer-based neural network, derived from BERT-style models, but optimized to be smaller and faster


In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Computing embeddings for all recipes...")
recipe_embeddings = sbert_model.encode(
    all_recipes['ingredient_text'].tolist(),
    show_progress_bar=True,
    batch_size=32
)
print(f"\nSBERT embeddings shape: {recipe_embeddings.shape}")

### Word2Vec

In [ ]:
sentences = []
for lst in all_recipes['ingredients_list']:
    # Flatten: split each ingredient string into words, then collect all words for this recipe
    recipe_words = []
    for phrase in lst:
        # Split by space
        words = phrase.split()
        recipe_words.extend(words)
    sentences.append(recipe_words)

# Train Word2Vec on these tokenised sentences
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=5, workers=4, sg=1)

In [ ]:
# Create recipe average vectors for Word2Vec
recipe_w2v_vectors = []
for tokens in sentences:
    vecs = [w2v_model.wv[w] for w in tokens if w in w2v_model.wv]
    if vecs:
        recipe_w2v_vectors.append(np.mean(vecs, axis=0))
    else:
        recipe_w2v_vectors.append(np.zeros(100))
recipe_w2v_vectors = np.array(recipe_w2v_vectors)
print(f"Word2Vec recipe vectors shape: {recipe_w2v_vectors.shape}")

In [ ]:
# Similarity test
print(w2v_model.wv.most_similar("yogurt", topn=5))
print(w2v_model.wv.most_similar("zucchini", topn=5))

## Dietry restriction mapping

### Approach 1 · Rule-Based Classification (Keyword Blacklist)

Each dietary label is defined by a set of **forbidden ingredient keywords**.  
A recipe earns a label when **none** of its cleaned ingredients match that label's blacklist.  
This is fast, fully transparent, and works well for the ~14 k recipe scale.

In [ ]:
# Dietary blacklists
# Keys appear in cleaned ingredient tokens (lower-case, lemmatised)
DIETARY_RULES = {
    "Gluten-Free": [
        "wheat", "flour", "barley", "rye", "oat", "semolina", "spelt",
        "breadcrumb", "soy source","bread", "pasta", "noodle", "cracker",
        "crouton", "couscous", "bulgur", "farro", "triticale", "malt"
    ],
    "Dairy-Free": [
        "milk", "cheese", "butter", "cream", "yogurt", "yoghurt", "whey",
        "lactose", "casein", "ghee", "kefir", "custard", "cheddar",
        "mozzarella", "parmesan", "brie", "ricotta", "sour cream",
        "half-and-half", "condensed milk", "evaporated milk", "buttermilk"
    ],
    "Vegan": [
        "meat", "chicken", "beef", "pork", "bacon", "lamb", "turkey", "duck",
        "bacon", "ham", "sausage", "salami", "pepperoni", "lard",
        "fish", "salmon", "tuna", "shrimp", "prawn", "crab", "lobster",
        "scallop", "anchovy", "sardine", "clam", "oyster", "mussel",
        "egg", "milk", "cheese", "butter", "cream", "yogurt", "honey",
        "gelatin", "whey", "casein", "ghee", "buttermilk", "mayo",
        "mayonnaise", "worcestershire"
    ],
    "Vegetarian": [
        "meat", "chicken", "beef", "pork", "bacon", "lamb", "turkey", "duck",
        "bacon", "ham", "sausage", "salami", "pepperoni", "lard",
        "fish", "salmon", "tuna", "shrimp", "prawn", "crab", "lobster",
        "scallop", "anchovy", "sardine", "clam", "oyster", "mussel",
        "gelatin", "rennet", "lard"
    ],
}

def classify_rule_based(ingredients_list):
    """Return dietary labels that apply to a recipe based on keyword blacklists."""
    text = " ".join(ingredients_list).lower()
    labels = []
    for diet, forbidden in DIETARY_RULES.items():
        if not any(word in text for word in forbidden):
            labels.append(diet)
    return labels if labels else ["None"]

# Apply to the dataset
all_recipes["dietary_rule"] = all_recipes["ingredients_list"].apply(classify_rule_based)

# Explode for easy counting
flat = [label for labels in all_recipes["dietary_rule"] for label in labels]
print("Rule-Based label distribution:")
for label, count in Counter(flat).most_common():
    pct = 100 * count / len(all_recipes)
    print(f"  {label:<15} {count:>5} recipes  ({pct:.1f}%)")

### Approach 2 · Embedding Similarity (uses your MiniLM embeddings)

Each dietary label is described in natural language.  
We encode those descriptions with MiniLM and then compute **cosine similarity**  
between each recipe's embedding and each label prototype.  
A recipe gets a label when its similarity exceeds a calibrated threshold.

In [ ]:
from sentence_transformers import util
import torch

# Label prototype descriptions
label_descriptions = {
    "Gluten-Free": (
        "a recipe that contains no wheat, flour, gluten, barley, rye, oats, "
        "semolina, bread, pasta or any grain containing gluten"
    ),
    "Dairy-Free": (
        "a recipe that contains no milk, cheese, butter, cream, yogurt, "
        "lactose, whey, casein or any dairy product"
    ),
    "Vegan": (
        "a fully plant-based recipe with no meat, chicken, beef, poultry, seafood, fish, "
        "egg, dairy, honey or any other animal product"
    ),
    "Vegetarian": (
        "a recipe that contains no meat, poultry, fish or seafood "
        "but may include eggs and dairy products"
    ),
}

# Force CPU to keep everything on the same device as recipe_embeddings
print("Encoding label prototypes...")
label_embeddings = {
    label: sbert_model.encode(
        desc,
        convert_to_tensor=True,
        normalize_embeddings=True,
        device="cpu"          # ← explicit CPU
    )
    for label, desc in label_descriptions.items()
}

# Ensure recipe_embeddings tensor is also on CPU
recipe_tensor = torch.tensor(recipe_embeddings).cpu()   # ← explicit CPU

# Cosine similarities
similarities = {}
for label, label_emb in label_embeddings.items():
    sims = util.cos_sim(recipe_tensor, label_emb.cpu()).squeeze(1)  # ← .cpu()
    similarities[label] = sims.numpy()

# Build a similarity DataFrame for inspection
sim_df = pd.DataFrame(similarities, index=all_recipes.index)
print("\nSimilarity score statistics (cosine sim to label prototypes):")
print(sim_df.describe().round(3))


In [ ]:
# Calibrate thresholds by inspecting similarity distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

# Use the rule-based column as soft ground truth to find good thresholds
suggested_thresholds = {}

for i, label in enumerate(label_descriptions.keys()):
    ax = axes[i]
    sims = sim_df[label].values

    # Recipes flagged as this label by rule-based approach
    positive_mask = all_recipes["dietary_rule"].apply(lambda x: label in x)

    ax.hist(sims[~positive_mask], bins=50, alpha=0.6, color="salmon",   label="Not " + label, density=True)
    ax.hist(sims[positive_mask],  bins=50, alpha=0.6, color="steelblue", label=label,          density=True)

    # Simple threshold: midpoint between the two group means
    mean_pos = sims[positive_mask].mean()  if positive_mask.sum() > 0 else 0
    mean_neg = sims[~positive_mask].mean() if (~positive_mask).sum() > 0 else 0
    threshold = (mean_pos + mean_neg) / 2
    suggested_thresholds[label] = round(float(threshold), 3)

    ax.axvline(threshold, color="black", linestyle="--", linewidth=1.5, label=f"Threshold={threshold:.3f}")
    ax.set_title(label, fontsize=12)
    ax.set_xlabel("Cosine Similarity")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

plt.suptitle("Similarity Score Distributions — Rule-Based Positives vs Negatives", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("dietary_similarity_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close()

print("\nSuggested thresholds (auto-calibrated):")
for label, t in suggested_thresholds.items():
    print(f"  {label:<15} {t}")


In [ ]:
#Classify using calibrated thresholds
THRESHOLDS = suggested_thresholds   # swap in manual values if you want to tune

def classify_by_embedding(row_sims: dict) -> list:
    labels = [label for label, t in THRESHOLDS.items() if row_sims[label] >= t]
    return labels if labels else ["None"]

# Apply row-wise
all_recipes["dietary_embed"] = [
    classify_by_embedding({label: sim_df[label].iloc[i] for label in THRESHOLDS})
    for i in range(len(all_recipes))
]

flat_embed = [l for labels in all_recipes["dietary_embed"] for l in labels]
print("Embedding-based label distribution:")
for label, count in Counter(flat_embed).most_common():
    pct = 100 * count / len(all_recipes)
    print(f"  {label:<15} {count:>5} recipes  ({pct:.1f}%)")


### Approach 3 · Hybrid (Rule-Based ∩ Embedding)

A recipe earns a label only when **both** methods agree.  
This dramatically reduces false positives: rules catch clear violations,  
embeddings add semantic context for edge cases.

In [ ]:
def classify_hybrid(rule_labels: list, embed_labels: list) -> list:
    """Keep only labels confirmed by both approaches."""
    combined = set(rule_labels) & set(embed_labels)
    # 'None' is only valid if both agree there are no labels
    if not combined or combined == {"None"}:
        return ["None"]
    return sorted(combined - {"None"})

all_recipes["dietary_preference"] = [
    classify_hybrid(r, e)
    for r, e in zip(all_recipes["dietary_rule"], all_recipes["dietary_embed"])
]

flat_hybrid = [l for labels in all_recipes["dietary_preference"] for l in labels]
print("Final Hybrid label distribution:")
for label, count in Counter(flat_hybrid).most_common():
    pct = 100 * count / len(all_recipes)
    print(f"  {label:<15} {count:>5} recipes  ({pct:.1f}%)")


In [ ]:
import matplotlib.patches as mpatches

# 1. Bar chart — label frequency
label_order   = ["Gluten-Free", "Dairy-Free", "Vegetarian", "Vegan", "None"]
label_counts  = {l: flat_hybrid.count(l) for l in label_order}

colors = ["#4CAF50", "#2196F3", "#FF9800", "#9C27B0", "#9E9E9E"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(label_order, [label_counts[l] for l in label_order],
             color=colors, edgecolor="white")
for i, l in enumerate(label_order):
    axes[0].text(label_counts[l] + 30, i, str(label_counts[l]), va="center", fontsize=10)
axes[0].set_xlabel("Number of Recipes")
axes[0].set_title("Dietary Preference Distribution\n(Hybrid Rule + Embedding)")
axes[0].invert_yaxis()

#2. Agreement matrix — rule vs embed
agreement_data = {}
for label in ["Gluten-Free", "Dairy-Free", "Vegetarian", "Vegan"]:
    rule_pos  = all_recipes["dietary_rule"].apply(lambda x: label in x)
    embed_pos = all_recipes["dietary_embed"].apply(lambda x: label in x)
    agreement_data[label] = {
        "Both":        int((rule_pos  & embed_pos).sum()),
        "Rule only":   int((rule_pos  & ~embed_pos).sum()),
        "Embed only":  int((~rule_pos & embed_pos).sum()),
        "Neither":     int((~rule_pos & ~embed_pos).sum()),
    }

categories = ["Both", "Rule only", "Embed only", "Neither"]
cat_colors  = ["#4CAF50", "#FF9800", "#2196F3", "#9E9E9E"]
x = np.arange(len(agreement_data))
bar_w = 0.2

for j, (cat, col) in enumerate(zip(categories, cat_colors)):
    vals = [agreement_data[lbl][cat] for lbl in agreement_data]
    axes[1].bar(x + j * bar_w, vals, width=bar_w, label=cat, color=col, edgecolor="white")

axes[1].set_xticks(x + bar_w * 1.5)
axes[1].set_xticklabels(list(agreement_data.keys()))
axes[1].set_ylabel("Number of Recipes")
axes[1].set_title("Rule vs. Embedding Agreement by Label")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig("dietary_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print("Plots saved.")


In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
all_recipes[["name","ingredients_list", "dietary_rule", "dietary_embed", "dietary_preference"]][:10]


Decision: use rule based - dietary_rule

## EDA

### Ingredients

In [ ]:
# Flatten all ingredient lists into one big list of phrases
all_ingredients = []
for lst in all_recipes['ingredients_list']:
    all_ingredients.extend(lst)

# Count unique phrases
unique_ingredients = set(all_ingredients)
print(f"Total number of unique cleaned ingredient phrases: {len(unique_ingredients)}")

In [ ]:
# Get top 15 ingredients
ingredient_freq = Counter(all_ingredients)
top = ingredient_freq.most_common(15)
top_ingredients, top_counts = zip(*top)

# Plot
plt.figure(figsize=(10, 6))
plt.barh(range(len(top_ingredients)), top_counts, color='#5B8DB8', edgecolor='white')
plt.yticks(range(len(top_ingredients)), top_ingredients)
plt.xlabel('Number of recipes')
plt.title('Top 15 Most Frequent Ingredients')
plt.gca().invert_yaxis()  # highest at top
plt.tight_layout()
plt.show()

In [ ]:
print("Top 15 ingredients:")
for ing, count in top:
    print(f"  {ing}: {count}")

In [ ]:
# Calculate number of ingredients per recipe
ingredient_counts = all_recipes['ingredients_list'].apply(len)

# Plot histogram
plt.figure(figsize=(10, 5))
plt.hist(ingredient_counts, bins=30, edgecolor='black', color='steelblue')
plt.xlabel('Number of ingredients per recipe')
plt.ylabel('Number of recipes')
plt.title('Distribution of Recipe Complexity (Ingredient Count)')
plt.axvline(ingredient_counts.median(), color='red', linestyle='--', label=f'Median = {ingredient_counts.median():.0f}')
plt.legend()
plt.show()

In [ ]:
print(f"Mean ingredients per recipe: {ingredient_counts.mean():.1f}")
print(f"Median: {ingredient_counts.median():.0f}")
print(f"Range: {ingredient_counts.min()} – {ingredient_counts.max()}")

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

# --- Top ingredients ---
ingredient_freq = Counter(all_ingredients)
top = ingredient_freq.most_common(15)
top_ingredients, top_counts = zip(*top)

# --- Ingredient counts per recipe ---
ingredient_counts = all_recipes['ingredients_list'].apply(len)

# --- Create subplots ---
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

# =========================
# Left: Top ingredients
# =========================
axes[0].barh(
    range(len(top_ingredients)),
    top_counts,
    color='#5B8DB8',
    edgecolor='white'
)

axes[0].set_yticks(range(len(top_ingredients)))
axes[0].set_yticklabels(top_ingredients)
axes[0].set_xlabel('Number of Recipes')
axes[0].set_title('Top 15 Most Frequent Ingredients', fontsize=10, fontweight='bold')
axes[0].invert_yaxis()

# =========================
# Right: Recipe complexity
# =========================
axes[1].hist(
    ingredient_counts,
    bins=30,
    edgecolor='black',
    color='steelblue'
)

axes[1].axvline(
    ingredient_counts.median(),
    color='red',
    linestyle='--',
    label=f'Median = {ingredient_counts.median():.0f}'
)

axes[1].set_xlabel('Number of Ingredients per Recipe')
axes[1].set_ylabel('Number of Recipes')
axes[1].set_title('Distribution of Recipe Complexity', fontsize=10, fontweight='bold')
axes[1].legend()

# --- Layout ---
plt.tight_layout()
plt.show()

### Servings

In [ ]:
top_servings = all_recipes['servings'].value_counts().head(12).sort_index()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(top_servings.index.astype(str), top_servings.values, color='steelblue', edgecolor='white')
ax.set_title('Frequency of Serving Sizes')
ax.set_xlabel('Servings')
sns.despine()
plt.tight_layout()
plt.show()

### Average Ratings

In [ ]:
# Summary statistics
print("Before dropping low-rated (< 3.5)")
print(f"Count:  {len(ratings_before)}")
print(f"Mean:   {ratings_before.mean():.2f}")
print(f"Median: {ratings_before.median():.2f}")
print(f"Min:    {ratings_before.min():.2f}")
print(f"Max:    {ratings_before.max():.2f}")

print("\nAfter dropping low-rated (< 3.5)")
print(f"Count:  {len(ratings_after)}")
print(f"Mean:   {ratings_after.mean():.2f}")
print(f"Median: {ratings_after.median():.2f}")
print(f"Min:    {ratings_after.min():.2f}")
print(f"Max:    {ratings_after.max():.2f}")

In [ ]:
# Plot: Histogram + Boxplot side by side
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Before: histogram
axes[0,0].hist(ratings_before, bins=25, edgecolor='black', color='steelblue', alpha=0.7)
axes[0,0].axvline(3.5, color='red', linestyle='--', linewidth=2, label='Threshold = 3.5')
axes[0,0].set_title('Before dropping (rating raw)')
axes[0,0].set_xlabel('Average rating')
axes[0,0].set_ylabel('Number of recipes')
axes[0,0].legend()

# After: histogram
axes[0,1].hist(ratings_after, bins=20, edgecolor='black', color='seagreen', alpha=0.7)
axes[0,1].set_title('After dropping rating < 3.5')
axes[0,1].set_xlabel('Average rating')
axes[0,1].set_ylabel('Number of recipes')

# Before: boxplot
axes[1,0].boxplot(ratings_before, vert=True, patch_artist=True,
                  boxprops=dict(facecolor='steelblue'))
axes[1,0].axhline(3.5, color='red', linestyle='--', linewidth=2, label='Threshold = 3.5')
axes[1,0].set_title('Before dropping (boxplot)')
axes[1,0].set_ylabel('Average rating')
axes[1,0].legend()

# After: boxplot
axes[1,1].boxplot(ratings_after, vert=True, patch_artist=True,
                  boxprops=dict(facecolor='seagreen'))
axes[1,1].set_title('After dropping (boxplot)')
axes[1,1].set_ylabel('Average rating')

plt.tight_layout()
plt.show()

### Dietary Labels

In [ ]:
# Define order and colors (matching your style)
label_order = ["Gluten-Free", "Dairy-Free", "Vegetarian", "Vegan", "None"]
colors = {"Gluten-Free": "#4CAF50", "Dairy-Free": "#2196F3",
          "Vegetarian": "#FF9800", "Vegan": "#9C27B0", "None": "#9E9E9E"}

# Count occurrences (each recipe counted once per label it has)
label_counts = {label: 0 for label in label_order}
for labels in all_recipes['dietary_rule']:
    for label in labels:
        if label in label_counts:
            label_counts[label] += 1

# Sort by count (largest to smallest)
sorted_labels = sorted(label_counts.items(), key=lambda x: x[1], reverse=True)
labels_sorted = [item[0] for item in sorted_labels]
counts_sorted = [item[1] for item in sorted_labels]
colors_sorted = [colors[l] for l in labels_sorted]

# Create horizontal bar chart
fig, ax = plt.subplots(figsize=(6, 3))

bars = ax.barh(labels_sorted, counts_sorted,
               color=colors_sorted, edgecolor='white', linewidth=1.5)

# Add value labels
for i, (label, count) in enumerate(zip(labels_sorted, counts_sorted)):
    ax.text(count + 50, i, str(count), va='center', fontsize=9)

ax.set_xlabel('Number of Recipes', fontsize=9)
ax.set_title('Rule‑Based Dietary Label Distribution', fontsize=10, fontweight='bold')
sns.despine()

plt.tight_layout()
plt.show()

# Print the sorted order for reference
print("\nSorted order (largest to smallest):")
for label, count in sorted_labels:
    pct = 100 * count / len(all_recipes)
    print(f"  {label}: {count} ({pct:.1f}%)")

## Evaluation of Embeddings

### Precision & Coverage

In [ ]:
TEST_QUERIES = [
    # Scenario 1: Common, easy ingredients
    {"id": "Q01", "ingredients": ["chicken", "garlic", "lemon", "olive oil"],
     "scenario": "common pantry"},

    # Scenario 2: Sparse / unusual combination
    {"id": "Q02", "ingredients": ["anchovies", "kimchi", "quinoa"],
     "scenario": "unusual combo"},

    # Scenario 3: Dietary constraint implied
    {"id": "Q03", "ingredients": ["tofu", "soy sauce", "ginger", "bok choy"],
     "scenario": "likely vegan"},

    # Scenario 4: Very few ingredients
    {"id": "Q04", "ingredients": ["eggs", "butter"],
     "scenario": "minimal input"},

    # Scenario 5: Many ingredients
    {"id": "Q05", "ingredients": ["beef", "onion", "carrot", "celery",
                                   "tomato", "red wine", "thyme", "bay leaf"],
     "scenario": "rich input"},

    # Scenario 6: Ingredients with no obvious match
    {"id": "Q06", "ingredients": ["dragonfruit", "miso", "cornmeal"],
     "scenario": "low coverage expected"},

    # Scenario 7: Expiring dairy
    {"id": "Q07", "ingredients": ["milk", "cream", "spinach", "garlic", "pasta"],
     "scenario": "expiring dairy"},

    # Scenario 8: Overripe fruit
    {"id": "Q08", "ingredients": ["banana", "strawberries", "yogurt", "honey", "oats"],
     "scenario": "overripe fruit"},

    # Scenario 9: Wilting vegetables
    {"id": "Q09", "ingredients": ["lettuce", "cucumber", "avocado", "chicken", "lemon"],
     "scenario": "wilting produce"},

    # Scenario 10: Leftover cooked meat
    {"id": "Q10", "ingredients": ["roast chicken", "rice", "egg", "peas", "soy sauce"],
     "scenario": "leftover protein"},

    # Scenario 11: Bread going stale
    {"id": "Q11", "ingredients": ["bread", "eggs", "milk", "cinnamon", "sugar"],
     "scenario": "stale carbs"},

    # Scenario 12: Herbs about to spoil
    {"id": "Q12", "ingredients": ["basil", "parsley", "garlic", "olive oil", "parmesan"],
     "scenario": "fresh herbs excess"},

    # Scenario 13: Expiring seafood
    {"id": "Q13", "ingredients": ["salmon", "lemon", "dill", "cream", "potatoes"],
     "scenario": "expiring seafood"},

    # Scenario 14: Partial pantry leftovers
    {"id": "Q14", "ingredients": ["coconut milk", "chickpeas", "tomato", "garlic", "spinach"],
     "scenario": "partial pantry leftovers"},

    # Scenario 15: Cheese leftovers
    {"id": "Q15", "ingredients": ["brie", "bread", "apple", "walnuts", "honey"],
     "scenario": "leftover cheese"},

    # Common / everyday
    {"id": "Q16", "ingredients": ["pasta", "tomato", "basil", "parmesan"],
     "scenario": "classic Italian"},

    {"id": "Q17", "ingredients": ["egg", "butter", "flour", "sugar", "milk"],
     "scenario": "basic baking"},

    {"id": "Q18", "ingredients": ["potato", "onion", "cheese"],
     "scenario": "simple comfort food"},

    {"id": "Q19", "ingredients": ["rice", "soy sauce", "sesame oil", "carrot", "egg"],
     "scenario": "Asian"},

    # Seasonal Leftovers

    {"id": "Q20", "ingredients": ["turkey", "cranberry", "sweet potato", "sage"],
     "scenario": "holiday leftovers"},

    {"id": "Q21", "ingredients": ["pumpkin", "cinnamon", "nutmeg", "cream"],
     "scenario": "autumn seasonal"},

    # Rich input
    {"id": "Q22", "ingredients": ["chicken", "coconut milk", "lemongrass",
                                   "ginger", "lime", "chili", "fish sauce", "coriander"],
     "scenario": "rich Thai-style"},

    # Vegan
    {"id": "Q23", "ingredients": ["lentil", "carrot", "onion", "vegetable broth", "turmeric"],
     "scenario": "vegan soup"},

    {"id": "Q24", "ingredients": ["avocado", "lime", "tomato", "red onion", "coriander"],
     "scenario": "vegan fresh"},

    # Vegetarian
    {"id": "Q25", "ingredients": ["egg", "spinach", "feta cheese", "onion"],
     "scenario": "vegetarian with dairy"},

    {"id": "Q26", "ingredients": ["mushroom", "cream", "thyme", "parmesan", "pasta"],
     "scenario": "vegetarian rich"},

    # Gluten-free
    {"id": "Q27", "ingredients": ["chicken", "sweet potato", "paprika", "olive oil"],
     "scenario": "gluten-free simple"},

    {"id": "Q28", "ingredients": ["quinoa", "black bean", "corn", "lime", "avocado"],
     "scenario": "gluten-free grain bowl"},

    # Dairy-free
    {"id": "Q29", "ingredients": ["coconut milk", "mango", "lime", "chili"],
     "scenario": "dairy-free tropical"},

    {"id": "Q30", "ingredients": ["tuna", "olive oil", "caper", "tomato", "garlic"],
     "scenario": "dairy-free Mediterranean"},
]


In [ ]:
def prepare_query(query_ingredients):
    """
    Convert query ingredient list into:
      - tokens: list of cleaned words (for TF‑IDF and Word2Vec)
      - query_str: space‑joined cleaned words (for SBERT)
    Uses the same cleaning function as recipe ingredients.
    """
    cleaned_parts = []
    for ing in query_ingredients:
        cleaned = clean_ingredient(ing)
        if cleaned:
            cleaned_parts.append(cleaned)

    tokens = []
    for part in cleaned_parts:
        tokens.extend(part.split())

    query_str = " ".join(tokens)
    return tokens, query_str

def retrieve_tfidf(query_ingredients, top_k=3):
    tokens, _ = prepare_query(query_ingredients)
    query_vec  = vectorizer.transform([tokens])          # pass-through expects list
    sims       = sklearn_cosine(query_vec, tfidf_matrix).flatten()
    top_idx    = np.argsort(sims)[-top_k:][::-1]
    return [(i, all_recipes.iloc[i]['ingredients_list'], sims[i]) for i in top_idx]

def retrieve_sbert(query_ingredients, top_k=3):
    _, query_str = prepare_query(query_ingredients)
    query_emb    = sbert_model.encode([query_str])             # mirrors ingredient_text
    sims         = sklearn_cosine(query_emb, recipe_embeddings).flatten()
    top_idx      = np.argsort(sims)[-top_k:][::-1]
    return [(i, all_recipes.iloc[i]['ingredients_list'], sims[i]) for i in top_idx]

def retrieve_word2vec(query_ingredients, top_k=3):
    tokens, _ = prepare_query(query_ingredients)
    vecs      = [w2v_model.wv[w] for w in tokens if w in w2v_model.wv]
    query_vec = np.mean(vecs, axis=0) if vecs else np.zeros(w2v_model.vector_size)
    sims      = sklearn_cosine([query_vec], recipe_w2v_vectors).flatten()
    top_idx   = np.argsort(sims)[-top_k:][::-1]
    return [(i, all_recipes.iloc[i]['ingredients_list'], sims[i]) for i in top_idx]

In [ ]:
def ingredient_coverage(user_ingredients, recipe_ingredients_list):
    """
    User has ["chicken","garlic","lemon"]
    Recipe word set contains {"chicken","salt","garlic"}
    Matches = {"chicken","garlic"} → 2 matches → coverage = 2/3 = 0.667
    """
    if not user_ingredients:
        return 1.0
    # Normalise user ingredients: lower case, strip spaces
    user_clean = set(ing.lower().strip() for ing in user_ingredients)
    # Collect all words from recipe ingredients
    recipe_words = set()
    for phrase in recipe_ingredients_list:
        for word in phrase.lower().split():
            recipe_words.add(word)
    # Count how many user ingredients appear as whole words
    matched = sum(1 for ing in user_clean if ing in recipe_words)
    return matched / len(user_ingredients)

In [ ]:
def evaluate_models(test_queries, models_dict, top_k=3):
    rows = []
    for q in test_queries:
        qid = q['id']
        user_ings = q['ingredients']
        for model_name, retrieve_func in models_dict.items():
            results = retrieve_func(user_ings, top_k=top_k)
            for rank, (idx, recipe_ings, score) in enumerate(results, start=1):
                cov = ingredient_coverage(user_ings, recipe_ings)
                rows.append({
                    'query_id': qid,
                    'model': model_name,
                    'rank': rank,
                    'recipe_index': idx,
                    'recipe_name': all_recipes.iloc[idx]['name'],
                    'recipe_ingredients': '; '.join(recipe_ings[:8]),  # show first 8 for readability
                    'coverage': cov,
                    'relevance': None  # to fill manually
                })
    return pd.DataFrame(rows)

# Define the models dictionary using your exact function names
models = {
    'tfidf': retrieve_tfidf,
    'sbert': retrieve_sbert,
    'word2vec': retrieve_word2vec
}

In [ ]:
# Run evaluation (assuming TEST_QUERIES is already defined as per your appendix)
eval_df = evaluate_models(TEST_QUERIES, models, top_k=3)

# Save to CSV for manual annotation
eval_df.to_csv('evaluation_results.csv', index=False)
print("Saved to evaluation_results.csv. Please open it, fill the 'relevance' column (1=relevant, 0=not relevant), then run the next script to compute final metrics.")

In [ ]:
evaluation_file_git_path = "https://raw.githubusercontent.com/JovitaBer/ML_2026_Practice/main/evaluation_results_annotated%20-%20evaluation_results_annotated.csv"
eval_df_a = pd.read_csv(evaluation_file_git_path, sep=",")


In [ ]:
eval_df_a.head(2)

In [ ]:
# Remove rows where relevance is still NaN (if any)
# eval_df_a = eval_df_a.dropna(subset=['relevance'])

# Precision@3 per query per model
precision_per_query = eval_df_a.groupby(['query_id', 'model']).apply(
    lambda g: g['relevance'].mean(), include_groups=False  # mean of the 3 ranks
).reset_index(name='precision@3')

# Average Precision@3 per model
avg_precision = precision_per_query.groupby('model')['precision@3'].mean().round(3)

# Average Ingredient Coverage per model (already computed for each row, average across all retrieved)
avg_coverage = eval_df_a.groupby('model')['coverage'].mean().round(3)

# Combine
comparison = pd.DataFrame({
    'Model': avg_precision.index,
    'Precision@3': avg_precision.values,
    'Ingridient Coverage': avg_coverage.values
})
print("Overall Retrieval Evaluation Results\n")
print(comparison)

In [ ]:
MODEL_COLORS = {
    "tfidf":    "#2E5496",   # blue
    "sbert":    "#4CAF50",   # green
    "word2vec": "#E07B39",   # amber
}
MODEL_LABELS = {"tfidf": "TF-IDF", "sbert": "SBERT", "word2vec": "Word2Vec"}

plt.rcParams.update({
    "font.family":      "sans-serif",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.grid":        True,
    "grid.color":       "#E5E5E5",
    "grid.linewidth":   0.7,
    "axes.axisbelow":   True,
})

# Compute metrics
eval_df_clean = eval_df_a.dropna(subset=["relevance"])

precision_per_query_group = (
    eval_df_clean
    .groupby(["scenario_group", "query_id", "model"])
    .apply(lambda g: g["relevance"].mean(), include_groups=False)
    .reset_index(name="precision@3")
)

group_precision = (
    precision_per_query_group
    .groupby(["scenario_group", "model"])["precision@3"]
    .mean().round(3)
)

group_coverage = (
    eval_df_clean
    .groupby(["scenario_group", "model"])["coverage"]
    .mean().round(3)
)

pivot_p = group_precision.unstack("model")
pivot_c = group_coverage.unstack("model")

models  = list(MODEL_COLORS.keys())
groups  = pivot_p.index.tolist()
x       = np.arange(len(groups))
n       = len(models)
width   = 0.25
offsets = np.linspace(-(n - 1) / 2, (n - 1) / 2, n) * width

# Figure layout
fig, axes = plt.subplots(2, 1, figsize=(14, 11), constrained_layout=True)
fig.suptitle("Retrieval Model Performance by Scenario Group",
             fontsize=16, fontweight="bold", y=1.01)

METRIC_CFG = [
    (axes[0], pivot_p, "Precision@3",        "Precision@3 (higher = better)", 1.05),
    (axes[1], pivot_c, "Ingredient Coverage", "Mean Ingredient Coverage (higher = better)", 1.05),
]

for ax, pivot, title, ylabel, ylim in METRIC_CFG:
    for i, model in enumerate(models):
        if model not in pivot.columns:
            continue
        vals  = pivot[model].values
        bars  = ax.bar(x + offsets[i], vals, width=width,
                       color=MODEL_COLORS[model], label=MODEL_LABELS[model],
                       edgecolor="white", linewidth=0.6, zorder=3)

        # Value labels on top of each bar
        for bar, val in zip(bars, vals):
            if not np.isnan(val):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.015,
                    f"{val:.2f}",
                    ha="center", va="bottom",
                    fontsize=7.5, color="#333333"
                )

    ax.set_title(title, fontsize=13, fontweight="bold", pad=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels(groups, rotation=25, ha="right", fontsize=9)
    ax.set_ylim(0, ylim)
    ax.legend(fontsize=9, framealpha=0.9, loc="upper right")

# Overall averages (horizontal reference lines)
for ax, pivot, *_ in METRIC_CFG:
    for model in models:
        if model not in pivot.columns:
            continue
        avg = pivot[model].mean()
        ax.axhline(avg, color=MODEL_COLORS[model],
                   linestyle="--", linewidth=0.9, alpha=0.45)

plt.savefig("model_performance_by_group.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print("Saved → model_performance_by_group.png")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

TOP_K     = 10
N_QUERIES = 200
SEED      = 42

rng       = np.random.default_rng(SEED)
query_idx = rng.choice(len(all_recipes), size=N_QUERIES, replace=False)

# Normalise matrices once for fast cosine similarity
tfidf_dense  = normalize(tfidf_matrix.toarray())
sbert_normed = normalize(recipe_embeddings)

def get_neighbours(idx, matrix, k=TOP_K):
    sims = matrix @ matrix[idx]
    sims[idx] = -1  # exclude self
    return np.argsort(sims)[::-1][:k]


# Metric 1: Dietary label agreement
scores_dietary = {"TF-IDF": [], "SBERT": []}

for idx in query_idx:
    q_labels = set(all_recipes.iloc[idx]["dietary_rule"])
    if q_labels == {"None"}:
        continue
    for label, matrix in [("TF-IDF", tfidf_dense), ("SBERT", sbert_normed)]:
        neighbours = get_neighbours(idx, matrix)
        agreements = []
        for n in neighbours:
            n_labels = set(all_recipes.iloc[n]["dietary_rule"])
            union = q_labels | n_labels
            agreements.append(len(q_labels & n_labels) / len(union) if union else 0.0)
        scores_dietary[label].append(np.mean(agreements))


# Metric 2: Intra-list diversity
scores_diversity = {"TF-IDF": [], "SBERT": []}

for idx in query_idx:
    for label, matrix in [("TF-IDF", tfidf_dense), ("SBERT", sbert_normed)]:
        neighbours = get_neighbours(idx, matrix)
        vecs = matrix[neighbours]
        sim_matrix = vecs @ vecs.T
        k = len(neighbours)
        upper = sim_matrix[np.triu_indices(k, k=1)]
        scores_diversity[label].append(1 - upper.mean())


# Print summary
all_scores = {
    "Dietary agreement": scores_dietary,
    "Diversity":         scores_diversity,
}

print(f"{'Metric':<22} {'TF-IDF':>8} {'SBERT':>8}  Winner")

for metric, scores in all_scores.items():
    t = np.mean(scores["TF-IDF"])
    s = np.mean(scores["SBERT"])
    winner = "TF-IDF" if t > s else "SBERT"
    print(f"{metric:<22} {t:>8.4f} {s:>8.4f}  {winner}")


# Plot
fig, axes = plt.subplots(1, 2, figsize=(9, 5))
titles  = ["Dietary label agreement", "Intra-list diversity"]
palette = {"TF-IDF": "#5B8DB8", "SBERT": "#E07B54"}

for ax, (metric, scores), title in zip(axes, all_scores.items(), titles):
    df = pd.DataFrame({
        "Score": scores["TF-IDF"] + scores["SBERT"],
        "Model": ["TF-IDF"] * len(scores["TF-IDF"]) + ["SBERT"] * len(scores["SBERT"]),
    })
    sns.boxplot(data=df, x="Model", y="Score", palette=palette, width=0.5, ax=ax,
                flierprops=dict(marker="o", markersize=2, alpha=0.4))
    for j, m in enumerate(["TF-IDF", "SBERT"]):
        mean_val = df[df["Model"] == m]["Score"].mean()
        ax.scatter(j, mean_val, color="white", s=60, zorder=5,
                   edgecolors="black", linewidths=1.2)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("")
    sns.despine(ax=ax)

plt.suptitle(f"TF-IDF vs SBERT  (top-{TOP_K}, n={N_QUERIES} queries)", fontsize=12)
plt.tight_layout()
plt.savefig("tfidf_vs_sbert.png", dpi=150, bbox_inches="tight")
plt.show()

# Small Language Model

This cell defines the retrieval pipeline:
- It first applies hard filters to narrow the candidate pool
- then uses SBERT to compute cosine similarity between the user's ingredients and all remaining recipes
- finally re-ranks the top candidates using
a hybrid score before returning the top 3 results.

In [ ]:
recipe_embeddings_arr = np.array(recipe_embeddings)

KNOWN_DIETS   = ["vegan", "vegetarian", "gluten-free", "dairy-free"]
SIM_WEIGHT    = 0.5
RATING_WEIGHT = 0.1
ING_WEIGHT    = 0.4


def get_ingredient_match_info(user_ings, recipe_ings):
    matched = []
    missing = []

    for ing in recipe_ings:
        if not ing.strip():
            continue
        ing_lower = ing.lower().strip()
        found = any(
            u.lower() in ing_lower or ing_lower in u.lower()
            for u in user_ings
        )
        if found:
            matched.append(ing.strip())
        else:
            missing.append(ing.strip())

    total = len([i for i in recipe_ings if i.strip()])
    score = len(matched) / total if total > 0 else 0
    return matched, missing, score


def recipe_contains_all(user_ings, recipe_ings):
    """Returns True only if every user ingredient appears in the recipe."""
    for u in user_ings:
        u_cleaned = clean_ingredient(u).lower().strip()
        if not u_cleaned:
            continue
        u_words = u_cleaned.split()
        found = any(
            any(word in ing.lower() for word in u_words)
            for ing in recipe_ings if ing.strip()
        )
        if not found:
            return False
    return True


def find_top_recipes_sbert(
    user_ingredients,
    dietary_restriction=None,
    max_total_time=None,
    servings=None,
    max_ingredients=None,
    top_n=3,
    candidate_pool=50,
):
    # Step 1: dietary filter
    if dietary_restriction:
        diet_lower   = dietary_restriction.lower().strip()
        matched_diet = next((d for d in KNOWN_DIETS if d in diet_lower), None)
        if matched_diet:
            pool = all_recipes[
                all_recipes["dietary_rule"]
                .astype(str).str.lower()
                .str.contains(matched_diet, na=False)
            ]
            print(f"  Dietary filter  : {len(pool)} {matched_diet} recipes")
        else:
            pool = all_recipes
            print(f"  Dietary filter  : '{dietary_restriction}' not recognised — searching all")
    else:
        pool = all_recipes
        print(f"  Dietary filter  : none — searching all {len(pool)} recipes")

    # Step 2: time filter
    if max_total_time is not None:
        before = len(pool)
        pool   = pool[pool["total_time"].isna() | (pool["total_time"] <= max_total_time)]
        print(f"  Time filter     : {before} -> {len(pool)} recipes (max {max_total_time} min)")

    # Step 3: servings filter
    if servings is not None:
        before = len(pool)
        pool   = pool[
            pool["servings"].isna() |
            ((pool["servings"] >= servings - 2) & (pool["servings"] <= servings + 2))
        ]
        print(f"  Servings filter : {before} -> {len(pool)} recipes ({servings} +/- 2)")

    # Step 4: max ingredients filter
    if max_ingredients is not None:
        before = len(pool)
        pool   = pool[
            pool["ingredients_list"].apply(
                lambda ings: len([i for i in ings if i.strip()]) <= max_ingredients
            )
        ]
        print(f"  Ingredient count : {before} -> {len(pool)} recipes (max {max_ingredients} ingredients)")

    # Step 5: hard ingredient filter, only keep recipes that contain ALL user ingredients
    before = len(pool)
    pool = pool[
        pool["ingredients_list"].apply(
            lambda ings: recipe_contains_all(user_ingredients, ings)
        )
    ]
    print(f"  Must-have filter : {before} -> {len(pool)} recipes (contain ALL your ingredients)")

    if len(pool) == 0:
        print("  No recipes contain ALL ingredients — showing best partial matches instead")
        pool = all_recipes

    pool_indices = [all_recipes.index.get_loc(idx) for idx in pool.index]

    # Step 6: SBERT similarity
    _, query_str    = prepare_query(user_ingredients)
    query_emb       = sbert_model.encode([query_str])
    pool_embeddings = recipe_embeddings_arr[pool_indices]
    sims            = sklearn_cosine(query_emb, pool_embeddings).flatten()
    n_candidates    = min(candidate_pool, len(pool))
    top_local_idx   = np.argsort(sims)[-n_candidates:][::-1]

    candidates               = pool.iloc[top_local_idx].copy()
    candidates["_sim_score"] = sims[top_local_idx]

    # Step 7: compute ingredient match score for each candidate
    ing_scores = []
    for _, row in candidates.iterrows():
        _, _, score = get_ingredient_match_info(
            user_ingredients, row.get("ingredients_list", [])
        )
        ing_scores.append(score)
    candidates["_ing_score"] = ing_scores

    # Step 8: hybrid score = similarity + ingredient match + rating
    candidates["_hybrid_score"] = (
        SIM_WEIGHT    * candidates["_sim_score"] +
        ING_WEIGHT    * candidates["_ing_score"] +
        RATING_WEIGHT * candidates["rating_norm"].fillna(0)
    )
    candidates = candidates.sort_values("_hybrid_score", ascending=False)

    return [candidates.iloc[i] for i in range(min(top_n, len(candidates)))]


print("Retrieval function ready.")
print(f"Ranking weights: {int(SIM_WEIGHT*100)}% similarity / {int(ING_WEIGHT*100)}% ingredient match / {int(RATING_WEIGHT*100)}% rating")

In [ ]:
!pip install nobodywho

This cell implements the Chef Bot conversational interface. It collects the user's ingredients and filters, retrieves the top 3 recipes using the pipeline defined above, prints them instantly via Python, and then opens a follow-up loop where the user can ask up to 5 questions answered by the Qwen3-1.7B SLM.

In [ ]:
from nobodywho import Chat
import re, time

CHAT_URL = "huggingface:NobodyWho/Qwen_Qwen3-1.7B-GGUF/Qwen_Qwen3-1.7B-Q4_K_M.gguf"

SYSTEM_PROMPT = """You are Chef Bot, a friendly recipe assistant.

RULES
1. You will receive recipe information and a user question.
2. Answer warmly and concisely in 2-3 sentences unless step-by-step instructions are asked for.
3. "recipe 1 / 2 / 3" always refers to the recipe with that number.
4. Never invent information. Missing or zero values -> "Not specified".
5. No internal thinking. Speak directly to the user.
"""

def strip_think(text):
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def label(title):
    print(f"\n  [ {title} ]\n")


def recipe_chatbot_sbert():

    print("\n  [ Chef Bot ]")
    print("  SBERT (all-MiniLM-L6-v2)  |  Qwen3-1.7B")
    print("  Type 'quit' to exit.")

    # Setup
    label("Setup")

    print("    Ingredients   >  (comma separated)")
    available = input("    > ").strip()
    if not available:
        available = "any"
    user_ings = [i.strip() for i in available.split(",")]

    print("\n    Restriction   >  [ vegan / vegetarian / gluten-free / dairy-free ]")
    print("                     Press Enter to skip.")
    dietary_raw = input("    > ").strip()

    print("\n    Max time      >  minutes  (Press Enter to skip)")
    time_input = input("    > ").strip()
    max_time   = int(time_input) if time_input.isdigit() else None

    print("\n    Servings      >  how many people  (Press Enter to skip)")
    serv_input = input("    > ").strip()
    servings   = int(serv_input) if serv_input.isdigit() else None

    # Searching
    label("Searching")

    t0          = time.time()
    top_recipes = find_top_recipes_sbert(
        user_ings,
        dietary_restriction=dietary_raw,
        max_total_time=max_time,
        servings=servings,
    )
    print(f"\n    Done in {time.time() - t0:.1f}s")

    # Build recipe context string (used by SLM for follow-up questions)
    recipe_context     = ""
    missing_per_recipe = []

    for i, row in enumerate(top_recipes, 1):
        prep       = row.get("prep_time")
        cook       = row.get("cook_time")
        prep_str   = f"{int(prep)} min" if prep and prep > 0 else "Not specified"
        cook_str   = f"{int(cook)} min" if cook and cook > 0 else "Not specified"
        rating_str = f"{row['avg_rating']:.1f} / 5" if row.get("avg_rating") else "Not specified"

        matched, missing, _ = get_ingredient_match_info(
            user_ings, row.get("ingredients_list", [])
        )
        missing_per_recipe.append(set(missing))

        matched_str = ", ".join(matched) if matched else "none"
        missing_str = ", ".join(missing) if missing else "none"

        recipe_context += f"\n{i}. {row['name']}\n"
        recipe_context += f"   Dietary      : {row.get('dietary_rule', 'N/A')}\n"
        recipe_context += f"   Prep time    : {prep_str}\n"
        recipe_context += f"   Cook time    : {cook_str}\n"
        recipe_context += f"   Rating       : {rating_str}\n"
        recipe_context += f"   You have     : {matched_str}\n"
        recipe_context += f"   You need     : {missing_str}\n"
        recipe_context += f"   Ingredients  :\n"
        for ing in row.get("ingredients_list", []):
            if ing.strip():
                recipe_context += f"     - {ing.strip()}\n"

    # Print all 3 recipes directly from Python
    label("Recipes")

    for i, row in enumerate(top_recipes, 1):
        prep       = row.get("prep_time")
        cook       = row.get("cook_time")
        prep_str   = f"{int(prep)} min" if prep and prep > 0 else "Not specified"
        cook_str   = f"{int(cook)} min" if cook and cook > 0 else "Not specified"
        rating_str = f"{row['avg_rating']:.1f} / 5" if row.get("avg_rating") else "Not specified"

        matched, missing, _ = get_ingredient_match_info(
            user_ings, row.get("ingredients_list", [])
        )

        matched_str = ", ".join(matched) if matched else "none"
        missing_str = ", ".join(missing) if missing else "Nothing, you have everything!"
        n_total     = len([x for x in row.get("ingredients_list", []) if x.strip()])
        n_matched   = len(matched)
        match_pct   = int(100 * n_matched / n_total) if n_total > 0 else 0

        print(f"  {i}. {row['name']}")
        print(f"     You already have {n_matched}/{n_total} ingredients ({match_pct}% match).")
        print(f"     Prep time   : {prep_str}")
        print(f"     Cook time   : {cook_str}")
        print(f"     Rating      : {rating_str}")
        print(f"     You have    : {matched_str}")
        print(f"     You need    : {missing_str}")
        print(f"     Ingredients :")
        for ing in row.get("ingredients_list", []):
            if ing.strip():
                print(f"       - {ing.strip()}")
        print()

    # Links
    label("Links")

    for i, row in enumerate(top_recipes, 1):
        print(f"    {i}. {row['name']}")
        print(f"       {row['url']}\n")


    # Follow-up loop: SML only used here for short answers
    MAX_Q = 5
    count = 0

    label("Questions")
    print("    e.g. 'Give me the instructions for recipe 1'")
    print("         'What can I substitute for butter in recipe 2?'")
    print("         'Which recipe is quickest?'\n")

    conversation_text = f"Recipe information:\n{recipe_context}\n\n"

    while count < MAX_Q:
        remaining  = MAX_Q - count
        user_input = input(f"    You  ({remaining} left)  >  ").strip()

        if not user_input:
            continue

        if user_input.lower() in ("quit", "exit", "q", "bye"):
            print("\n    Chef Bot  :  Happy cooking!\n")
            break

        count += 1

        context = conversation_text + f"User: {user_input}\n\nChef Bot:"

        label("Chef Bot")

        t4  = time.time()
        raw = ""
        for token in Chat(CHAT_URL, system_prompt=SYSTEM_PROMPT).ask(context):
            raw += token
        reply = strip_think(raw)

        print(reply)
        print(f"\n    ({time.time() - t4:.1f}s)\n")

        conversation_text += f"User: {user_input}\nChef Bot: {reply}\n\n"

        if count == MAX_Q:
            print("    Chef Bot  :  That is the 5-question limit for this session.")
            print("    Run the cell again to start a new search.\n")


recipe_chatbot_sbert()